## Imports & Variables 

In [34]:
import pandas as pd
import pygeohash as pgh
import datetime
import sys
import os
import requests
from io import StringIO
from datetime import datetime, time 
from pathlib import Path

root_path = os.path.abspath(os.path.join(os.getcwd(), ".."))
if root_path not in sys.path:
    sys.path.append(root_path)

from python.kp_index import update_kp_data
from python.geo_location import create_geohashes



## Data Cleaning & Enrichment for US UAP Reports 1940 - 2014 

### Read the UAP Dataset into a Pandas Dataframe
- "../data/raw/uap_original_dataset.csv"
- a few rows in this large dataset have an extra column of irrelevent data
    - the fix: usecols=range(0, 11)

In [66]:
uap_df = pd.read_csv("../data/raw/uap_original_dataset.csv", usecols=range(0, 11), low_memory=False) 

uap_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 88875 entries, 0 to 88874
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   datetime              88875 non-null  object 
 1   city                  88679 non-null  object 
 2   state                 81356 non-null  object 
 3   country               76314 non-null  object 
 4   shape                 85757 non-null  object 
 5   duration (seconds)    88873 non-null  object 
 6   duration (hours/min)  85772 non-null  object 
 7   comments              88749 non-null  object 
 8   date posted           88875 non-null  object 
 9   latitude              88875 non-null  object 
 10  longitude             88875 non-null  float64
dtypes: float64(1), object(10)
memory usage: 7.5+ MB


In [67]:
uap_df.head()

,datetime,city,state,country,shape,duration (seconds),duration (hours/min),comments,date posted,latitude,longitude
0,10/10/1949 20:30,san marcos,tx,us,cylinder,2700,45 minutes,This event took place in early fall around 194...,4/27/2004,29.8830556,-97.941111
1,10/10/1949 21:00,lackland afb,tx,NaN,light,7200,1-2 hrs,1949 Lackland AFB&#44 TX. Lights racing acros...,12/16/2005,29.38421,-98.581082
2,10/10/1955 17:00,chester (uk/england),NaN,gb,circle,20,20 seconds,Green/Orange circular disc over Chester&#44 En...,1/21/2008,53.2,-2.916667
3,10/10/1956 21:00,edna,tx,us,circle,20,1/2 hour,My older brother and twin sister were leaving ...,1/17/2004,28.9783333,-96.645833
4,10/10/1960 20:00,kaneohe,hi,us,light,900,15 minutes,AS a Marine 1st Lt. flying an FJ4B fighter/att...,1/22/2004,21.4180556,-157.803611


In [68]:
# At least one row in the 'datetime' col contains an invalid 24:00 for the time
# AI Use: Grok assistance in fixing any entries with 24:00 in formating datetime

# Make a copy of the original column
col = uap_df['datetime'].astype(str)

# Handle 24:00 cases
mask_24 = col.str.contains('24:00', na=False)
col = col.str.replace('24:00', '00:00', regex=False)

# Convert to real datetime
uap_df['datetime'] = pd.to_datetime(col, format='%m/%d/%Y %H:%M', errors='coerce')

# Fix the date rollover for 24:00
uap_df.loc[mask_24, 'datetime'] = uap_df.loc[mask_24, 'datetime'] + pd.Timedelta(days=1)

# Create the columns 
uap_df['datetime_formatted'] = uap_df['datetime'].dt.strftime('%Y-%m-%d %H:%M')   # yyyy-mm-dd hh:mm
uap_df['full_date']           = uap_df['datetime'].dt.strftime('%Y-%m-%d')         # yyyy-mm-dd only

### Create Dataframe with Only US sightings 

In [38]:
us_uap_df = uap_df[uap_df["country"].str.lower() == "us"].copy()
us_uap_df.head()

,datetime,city,state,country,shape,duration (seconds),duration (hours/min),comments,date posted,latitude,longitude,datetime_formatted,full_date
0,1949-10-10 20:30:00,san marcos,tx,us,cylinder,2700,45 minutes,This event took place in early fall around 194...,4/27/2004,29.8830556,-97.941111,1949-10-10 20:30,1949-10-10
3,1956-10-10 21:00:00,edna,tx,us,circle,20,1/2 hour,My older brother and twin sister were leaving ...,1/17/2004,28.9783333,-96.645833,1956-10-10 21:00,1956-10-10
4,1960-10-10 20:00:00,kaneohe,hi,us,light,900,15 minutes,AS a Marine 1st Lt. flying an FJ4B fighter/att...,1/22/2004,21.4180556,-157.803611,1960-10-10 20:00,1960-10-10
5,1961-10-10 19:00:00,bristol,tn,us,sphere,300,5 minutes,My father is now 89 my brother 52 the girl wit...,4/27/2007,36.5950000,-82.188889,1961-10-10 19:00,1961-10-10
7,1965-10-10 23:45:00,norwalk,ct,us,disk,1200,20 minutes,A bright orange color changing to reddish colo...,10/2/1999,41.1175000,-73.408333,1965-10-10 23:45,1965-10-10


In [39]:
us_uap_after_1940_df = us_uap_df[
    us_uap_df["datetime"] >= '1940-01-01'
    
].copy()

us_uap_after_1940_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 70276 entries, 0 to 88874
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   datetime              70276 non-null  datetime64[ns]
 1   city                  70276 non-null  object        
 2   state                 70276 non-null  object        
 3   country               70276 non-null  object        
 4   shape                 68047 non-null  object        
 5   duration (seconds)    70275 non-null  object        
 6   duration (hours/min)  68054 non-null  object        
 7   comments              70248 non-null  object        
 8   date posted           70276 non-null  object        
 9   latitude              70276 non-null  object        
 10  longitude             70276 non-null  float64       
 11  datetime_formatted    70276 non-null  object        
 12  full_date             70276 non-null  object        
dtypes: datetime64[ns](1),

### Find out how many null values remain

In [40]:
us_uap_after_1940_df.isnull().sum()

datetime                   0
city                       0
state                      0
country                    0
shape                   2229
duration (seconds)         1
duration (hours/min)    2222
comments                  28
date posted                0
latitude                   0
longitude                  0
datetime_formatted         0
full_date                  0
dtype: int64

### Fix null values in 'shape' column
- begin with finding unique values 

In [41]:
us_uap_after_1940_df['shape'].unique()

array(['cylinder', 'circle', 'light', 'sphere', 'disk', 'fireball',
       'unknown', 'oval', 'other', 'rectangle', 'chevron', 'formation',
       'triangle', 'cigar', nan, 'delta', 'changing', 'diamond', 'flash',
       'egg', 'teardrop', 'cone', 'cross', 'pyramid', 'round', 'flare',
       'hexagon', 'crescent', 'changed'], dtype=object)

### Assign 'shape' NA and Null values as 'unknown'

In [42]:
us_uap_after_1940_df['shape'] = us_uap_after_1940_df['shape'].fillna('unknown')
us_uap_after_1940_df.isnull().sum()

datetime                   0
city                       0
state                      0
country                    0
shape                      0
duration (seconds)         1
duration (hours/min)    2222
comments                  28
date posted                0
latitude                   0
longitude                  0
datetime_formatted         0
full_date                  0
dtype: int64

### fix durration (sec) null value - only one
- look at the row and decide to remove entry or fill in value .. 

In [43]:
us_uap_after_1940_df[us_uap_after_1940_df['duration (seconds)'].isnull()]

# airport sighting, unclear durration based on other values - drop row in place 
us_uap_after_1940_df.dropna(subset=['duration (seconds)'], inplace=True)

# check null value totals 
us_uap_after_1940_df.isnull().sum()

datetime                   0
city                       0
state                      0
country                    0
shape                      0
duration (seconds)         0
duration (hours/min)    2222
comments                  28
date posted                0
latitude                   0
longitude                  0
datetime_formatted         0
full_date                  0
dtype: int64

### remove 'duration hours / mins' 
- (null and wierd values + seconds column gives us duration and is more nomalized)

In [44]:
us_uap_after_1940_df.drop(columns = ['duration (hours/min)'], inplace=True)

# check null value totals 
us_uap_after_1940_df.isnull().sum()

datetime               0
city                   0
state                  0
country                0
shape                  0
duration (seconds)     0
comments              28
date posted            0
latitude               0
longitude              0
datetime_formatted     0
full_date              0
dtype: int64

### Fix null values in 'comments' column

In [45]:
# what do the rows look like? 
us_uap_after_1940_df[us_uap_after_1940_df['comments'].isnull()]

# fill with null comments with 'no comment' 
us_uap_after_1940_df['comments'] = us_uap_after_1940_df['comments'].fillna('no comments')

us_uap_after_1940_df.isnull().sum()

datetime              0
city                  0
state                 0
country               0
shape                 0
duration (seconds)    0
comments              0
date posted           0
latitude              0
longitude             0
datetime_formatted    0
full_date             0
dtype: int64

### convert 'duration (seconds)' to int
- some entries have non-integer values in them
- use regex in conversion to remove non-integers 

In [46]:
us_uap_after_1940_df['duration (seconds)'] = us_uap_after_1940_df['duration (seconds)'].astype(str).str.replace(r'[^0-9.]', '', regex=True)
us_uap_after_1940_df['duration (seconds)'] = us_uap_after_1940_df['duration (seconds)'].astype(float)
us_uap_after_1940_df['duration (seconds)'] = us_uap_after_1940_df['duration (seconds)'].round(0)
us_uap_after_1940_df['duration (seconds)'] = us_uap_after_1940_df['duration (seconds)'].astype(int)
us_uap_after_1940_df.head()

,datetime,city,state,country,shape,duration (seconds),comments,date posted,latitude,longitude,datetime_formatted,full_date
0,1949-10-10 20:30:00,san marcos,tx,us,cylinder,2700,This event took place in early fall around 194...,4/27/2004,29.8830556,-97.941111,1949-10-10 20:30,1949-10-10
3,1956-10-10 21:00:00,edna,tx,us,circle,20,My older brother and twin sister were leaving ...,1/17/2004,28.9783333,-96.645833,1956-10-10 21:00,1956-10-10
4,1960-10-10 20:00:00,kaneohe,hi,us,light,900,AS a Marine 1st Lt. flying an FJ4B fighter/att...,1/22/2004,21.4180556,-157.803611,1960-10-10 20:00,1960-10-10
5,1961-10-10 19:00:00,bristol,tn,us,sphere,300,My father is now 89 my brother 52 the girl wit...,4/27/2007,36.5950000,-82.188889,1961-10-10 19:00,1961-10-10
7,1965-10-10 23:45:00,norwalk,ct,us,disk,1200,A bright orange color changing to reddish colo...,10/2/1999,41.1175000,-73.408333,1965-10-10 23:45,1965-10-10


### Convert latitude to float type (longitude is already a float)

In [47]:
us_uap_after_1940_df['latitude'] = us_uap_after_1940_df['latitude'].astype(float)

us_uap_after_1940_df.dtypes

datetime              datetime64[ns]
city                          object
state                         object
country                       object
shape                         object
duration (seconds)             int64
comments                      object
date posted                   object
latitude                     float64
longitude                    float64
datetime_formatted            object
full_date                     object
dtype: object

### Convert 'full_date' to datetime + create 'year' and 'month' column

In [48]:
us_uap_after_1940_df['full_date'] = pd.to_datetime(us_uap_after_1940_df['full_date'])

us_uap_after_1940_df['year'] = us_uap_after_1940_df['full_date'].astype(str).str[:4]
us_uap_after_1940_df['year'] = us_uap_after_1940_df['year'].astype(int)

us_uap_after_1940_df['month'] = us_uap_after_1940_df['full_date'].astype(str).str[5:7]
us_uap_after_1940_df['month'] = us_uap_after_1940_df['month'].astype(int)

us_uap_after_1940_df.head()
us_uap_after_1940_df.dtypes

datetime              datetime64[ns]
city                          object
state                         object
country                       object
shape                         object
duration (seconds)             int64
comments                      object
date posted                   object
latitude                     float64
longitude                    float64
datetime_formatted            object
full_date             datetime64[ns]
year                           int64
month                          int64
dtype: object

# Enrich data even more with a 'season' column
- create season_groups dictionary
- map to 'month' value in each row

In [49]:
season_groups = {
    '01': 'winter', 
    '02': 'winter', 
    '03': 'spring', 
    '04': 'spring', 
    '05': 'spring',
    '06': 'summer',
    '07': 'summer', 
    '08': 'summer',
    '09': 'fall', 
    '10': 'fall',
    '11': 'fall',
    '12': 'winter',
}

us_uap_after_1940_df['season'] = us_uap_after_1940_df['month'].astype(str).str.zfill(2).map(season_groups)

us_uap_after_1940_df.head()

,datetime,city,state,country,shape,duration (seconds),comments,date posted,latitude,longitude,datetime_formatted,full_date,year,month,season
0,1949-10-10 20:30:00,san marcos,tx,us,cylinder,2700,This event took place in early fall around 194...,4/27/2004,29.883056,-97.941111,1949-10-10 20:30,1949-10-10,1949,10,fall
3,1956-10-10 21:00:00,edna,tx,us,circle,20,My older brother and twin sister were leaving ...,1/17/2004,28.978333,-96.645833,1956-10-10 21:00,1956-10-10,1956,10,fall
4,1960-10-10 20:00:00,kaneohe,hi,us,light,900,AS a Marine 1st Lt. flying an FJ4B fighter/att...,1/22/2004,21.418056,-157.803611,1960-10-10 20:00,1960-10-10,1960,10,fall
5,1961-10-10 19:00:00,bristol,tn,us,sphere,300,My father is now 89 my brother 52 the girl wit...,4/27/2007,36.595000,-82.188889,1961-10-10 19:00,1961-10-10,1961,10,fall
7,1965-10-10 23:45:00,norwalk,ct,us,disk,1200,A bright orange color changing to reddish colo...,10/2/1999,41.117500,-73.408333,1965-10-10 23:45,1965-10-10,1965,10,fall


### Add KP Index Data ('solar_kp_index', 'solar_ap_index' columns)
- Call update_kp_data function in module
- Read csv file into pandas df
- Ensure datetime is datetime 
- KP data is in UTZ and updated every 3 hours: for a given day, find the the data that matched to 9pm 21:00 US Central Time
- Create a 'full_date' column in the kp_9pm_us_cst_df to merge on.
- Merge to full_date to us uap sightings

In [50]:
update_kp_data()

   year  month  day  hour_start  hour_end  decimal_day_start  decimal_day_end  \
0  1932      1    1         0.0       1.5              0.000           0.0625   
1  1932      1    1         3.0       4.5              0.125           0.1875   
2  1932      1    1         6.0       7.5              0.250           0.3125   
3  1932      1    1         9.0      10.5              0.375           0.4375   
4  1932      1    1        12.0      13.5              0.500           0.5625   

      kp  ap  flag            datetime  
0  3.333  18     1 1932-01-01 00:00:00  
1  2.667  12     1 1932-01-01 03:00:00  
2  2.333   9     1 1932-01-01 06:00:00  
3  2.667  12     1 1932-01-01 09:00:00  
4  3.333  18     1 1932-01-01 12:00:00  


,year,month,day,hour_start,hour_end,decimal_day_start,decimal_day_end,kp,ap,flag,datetime
0,1932,1,1,0.0,1.5,0.000,0.0625,3.333,18,1,1932-01-01 00:00:00
1,1932,1,1,3.0,4.5,0.125,0.1875,2.667,12,1,1932-01-01 03:00:00
2,1932,1,1,6.0,7.5,0.250,0.3125,2.333,9,1,1932-01-01 06:00:00
3,1932,1,1,9.0,10.5,0.375,0.4375,2.667,12,1,1932-01-01 09:00:00
4,1932,1,1,12.0,13.5,0.500,0.5625,3.333,18,1,1932-01-01 12:00:00
...,...,...,...,...,...,...,...,...,...,...,...
276249,2026,7,17,3.0,4.5,34531.125,34531.1875,0.667,3,0,2026-07-17 03:00:00
276250,2026,7,17,6.0,7.5,34531.250,34531.3125,0.667,3,0,2026-07-17 06:00:00
276251,2026,7,17,9.0,10.5,34531.375,34531.4375,0.667,3,0,2026-07-17 09:00:00
276252,2026,7,17,12.0,13.5,34531.500,34531.5625,1.333,5,0,2026-07-17 12:00:00


In [51]:
kp_df = pd.read_csv("../data/processed/kp_index.csv", usecols=('datetime', 'kp', 'ap'))
kp_df.head(20)

,kp,ap,datetime
0,3.333,18,1932-01-01 00:00:00
1,2.667,12,1932-01-01 03:00:00
2,2.333,9,1932-01-01 06:00:00
3,2.667,12,1932-01-01 09:00:00
4,3.333,18,1932-01-01 12:00:00
5,2.667,12,1932-01-01 15:00:00
6,3.333,18,1932-01-01 18:00:00
7,3.333,18,1932-01-01 21:00:00
8,3.667,22,1932-01-02 00:00:00
9,3.667,22,1932-01-02 03:00:00


In [52]:
# ensure datetime column is a datetime type
kp_df['datetime'] = pd.to_datetime(kp_df['datetime'])
kp_df.dtypes

kp                 float64
ap                   int64
datetime    datetime64[ns]
dtype: object

In [53]:
# kp table includes values in utz (5-6 hours ahead of most US timezones)
# most uap sightings in the US happen late in the evening / close 9:00pm or 21:00 
# 03:00:00 or 3am UTZ would be the best match for most sightings .. which is 9pm central standard time 

# only 03:00 utz (it will have the next day's date, so we would need to convert to US time zome)
target_time = time(3, 0, 0)

# Filter for exactly 03:00 UTC
kp_3am_utz_df = kp_df[kp_df['datetime'].dt.time == target_time].copy()

# Convert to US Central Standard Time (CST = UTC-6)
kp_9pm_us_cst_df = kp_3am_utz_df.copy()
kp_9pm_us_cst_df['datetime'] = kp_9pm_us_cst_df['datetime'] - pd.Timedelta(hours=6)

kp_9pm_us_cst_df.head()



,kp,ap,datetime
1,2.667,12,1931-12-31 21:00:00
9,3.667,22,1932-01-01 21:00:00
17,3.333,18,1932-01-02 21:00:00
25,0.333,2,1932-01-03 21:00:00
33,0.000,0,1932-01-04 21:00:00


In [54]:
# Create a 'full_date' column in the kp_9pm_us_cst_df to merge on

kp_9pm_us_cst_df['full_date'] = kp_9pm_us_cst_df['datetime'].dt.strftime('%Y-%m-%d')
kp_9pm_us_cst_df['full_date'] = pd.to_datetime(kp_9pm_us_cst_df['full_date'])
kp_9pm_us_cst_df.head()

,kp,ap,datetime,full_date
1,2.667,12,1931-12-31 21:00:00,1931-12-31
9,3.667,22,1932-01-01 21:00:00,1932-01-01
17,3.333,18,1932-01-02 21:00:00,1932-01-02
25,0.333,2,1932-01-03 21:00:00,1932-01-03
33,0.000,0,1932-01-04 21:00:00,1932-01-04


In [55]:
# Merge to full_date to us uap sightings

us_uap_after_1940_df = pd.merge(us_uap_after_1940_df, kp_9pm_us_cst_df[['full_date','kp', 'ap']], on='full_date', how='left')

us_uap_after_1940_df.head()

,datetime,city,state,country,shape,duration (seconds),comments,date posted,latitude,longitude,datetime_formatted,full_date,year,month,season,kp,ap
0,1949-10-10 20:30:00,san marcos,tx,us,cylinder,2700,This event took place in early fall around 194...,4/27/2004,29.883056,-97.941111,1949-10-10 20:30,1949-10-10,1949,10,fall,2.667,12
1,1956-10-10 21:00:00,edna,tx,us,circle,20,My older brother and twin sister were leaving ...,1/17/2004,28.978333,-96.645833,1956-10-10 21:00,1956-10-10,1956,10,fall,2.667,12
2,1960-10-10 20:00:00,kaneohe,hi,us,light,900,AS a Marine 1st Lt. flying an FJ4B fighter/att...,1/22/2004,21.418056,-157.803611,1960-10-10 20:00,1960-10-10,1960,10,fall,3.667,22
3,1961-10-10 19:00:00,bristol,tn,us,sphere,300,My father is now 89 my brother 52 the girl wit...,4/27/2007,36.595000,-82.188889,1961-10-10 19:00,1961-10-10,1961,10,fall,1.667,6
4,1965-10-10 23:45:00,norwalk,ct,us,disk,1200,A bright orange color changing to reddish colo...,10/2/1999,41.117500,-73.408333,1965-10-10 23:45,1965-10-10,1965,10,fall,0.333,2


### Additional Cleanup
- create state_code column in uppercase to match bigfoot data and ERD
- rename columns to match ERD

In [56]:
# create state_code column in uppercase to match bigfoot data and ERD

us_uap_after_1940_df['state_code'] = us_uap_after_1940_df['state'].str.upper()

# rename to match ERD

us_uap_after_1940_df = us_uap_after_1940_df.rename(
    columns={
        'duration (seconds)': 'duration_secs',
        'kp': 'solar_kp_index', 
        'ap': 'solar_ap_index'
    }
)

### Group Shapes - Enrich Data with 'shape_group' column
- There are many unique 'shapes' reported, but many are similar in 'type'
    - example: 'fireball' and 'flash' are both 'light' types
- Create a shape_groups dictionary
    - Missing or unknown values will be grouped in the 'other' category 
- Map to 'shape' column 

In [57]:

# group shapes by types 

shape_groups = {
    'circle': 'round', 'sphere': 'round', 'disk': 'round', 
    'oval': 'round', 'round': 'round', 'crescent': 'round', 'dome': 'round',
    'egg': 'round', 'teardrop': 'round',
    
    'triangle': 'triangle', 'delta': 'triangle', 'chevron': 'triangle', 
    'pyramid': 'triangle', 'diamond': 'triangle',
    
    'cylinder': 'cigar', 'cigar': 'cigar',
    
    'light': 'light', 'fireball': 'light', 'flash': 'light', 'flare': 'light',
    
    'changing': 'changing', 'changed': 'changing', 'formation': 'changing', 'cone': 'changing',
    'cross': 'changing', 'hexagon': 'changing', 'rectangle': 'changing',
    
    'other': 'other', 
    'nan': 'other',
    'unknown': 'other',
    '': 'other'
}

us_uap_after_1940_df['shape_group'] = us_uap_after_1940_df['shape'].map(shape_groups)

us_uap_after_1940_df['shape_group'].value_counts()

shape_group
light       20796
round       20253
other       12139
triangle     8809
changing     5442
cigar        2836
Name: count, dtype: int64

### Check head and save to CSV in data/processed/ folder 

In [58]:
us_uap_after_1940_df.head()

,datetime,city,state,country,shape,duration_secs,comments,date posted,latitude,longitude,datetime_formatted,full_date,year,month,season,solar_kp_index,solar_ap_index,state_code,shape_group
0,1949-10-10 20:30:00,san marcos,tx,us,cylinder,2700,This event took place in early fall around 194...,4/27/2004,29.883056,-97.941111,1949-10-10 20:30,1949-10-10,1949,10,fall,2.667,12,TX,cigar
1,1956-10-10 21:00:00,edna,tx,us,circle,20,My older brother and twin sister were leaving ...,1/17/2004,28.978333,-96.645833,1956-10-10 21:00,1956-10-10,1956,10,fall,2.667,12,TX,round
2,1960-10-10 20:00:00,kaneohe,hi,us,light,900,AS a Marine 1st Lt. flying an FJ4B fighter/att...,1/22/2004,21.418056,-157.803611,1960-10-10 20:00,1960-10-10,1960,10,fall,3.667,22,HI,light
3,1961-10-10 19:00:00,bristol,tn,us,sphere,300,My father is now 89 my brother 52 the girl wit...,4/27/2007,36.595000,-82.188889,1961-10-10 19:00,1961-10-10,1961,10,fall,1.667,6,TN,round
4,1965-10-10 23:45:00,norwalk,ct,us,disk,1200,A bright orange color changing to reddish colo...,10/2/1999,41.117500,-73.408333,1965-10-10 23:45,1965-10-10,1965,10,fall,0.333,2,CT,round


In [59]:
# save to a csv copy before adding geohash and weather data

us_uap_after_1940_df.to_csv("../data/processed/us_uap_1940_v1.csv", index=False)

### Additional Enrichment Before Saving to Final

In [62]:
# addional enrichment? 

### Finished? Save to 'data/final/' folder 

In [61]:
us_uap_after_1940_df.to_csv("../data/final/uap_reports.csv", index=False)

## Data Cleaning & Enrichment for Bigfoot Dataset(s)

### Read the first Bigfoot Dataset 
- '../data/raw/bfro_locations.csv'

In [65]:
pd.set_option('display.max_columns', None) 

bigfoot_df = pd.read_csv('../data/raw/bfro_locations.csv', index_col=False)

bigfoot_df.shape
bigfoot_df.describe(include='all')
bigfoot_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4250 entries, 0 to 4249
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   index           4250 non-null   int64  
 1   number          4250 non-null   int64  
 2   title           4250 non-null   object 
 3   classification  4250 non-null   object 
 4   timestamp       4250 non-null   object 
 5   latitude        4250 non-null   float64
 6   longitude       4250 non-null   float64
dtypes: float64(2), int64(2), object(3)
memory usage: 232.6+ KB
